# HQNN-Topology: Full Demonstration

This notebook walks through all five phases of the HQNN simulation pipeline.

In [ ]:
import sys
sys.path.insert(0, '..')
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline

from hqnn import (
    HyperconnectedQuantumNetwork, NodeConfig,
    GroverSearch, ShorAlgorithm, VQEAlgorithm,
    TopologicalErrorCorrector, SurfaceCode,
    QuantumSlimeMoldOptimizer, QuantumBeamCage, BeamParameters,
    NoisePredictor, NoisePredictorTrainer,
    HybridAdaptiveController,
    plot_network_graph, plot_entanglement_matrix,
)

## Phase A: Quantum Algorithms


In [ ]:
grover = GroverSearch(n_qubits=4)
result = grover.run(targets=[11], noise_level=0.02)
print(f'Target probability: {result["final_target_probability"]:.4f}')
print(f'Measured: {result["measured"]} | Success: {result["success"]}')
plt.figure(figsize=(10,4))
plt.plot(result['target_probabilities'], 'b-o', markersize=5)
plt.title('Grover Search: P(target) per Iteration')
plt.xlabel('Iteration')
plt.ylabel('P(target)')
plt.grid(True, alpha=0.3)
plt.show()

## Phase B: Quantum Beam Cage


In [ ]:
params = BeamParameters(beam_waist=1.5e-6, wavelength=780e-9, power=8e-3)
cage = QuantumBeamCage(params)
cage_result = cage.simulate_atomic_motion(n_atoms=500, T_atom=1e-6, n_steps=200)
print(f'Trapped fraction: {cage_result["trapped_fraction"][-1]:.2%}')
print(f'Final coherence: {cage_result["coherence"][-1]:.4f}')

## Phase C: Network Visualization


In [ ]:
net = HyperconnectedQuantumNetwork(n_nodes=9, connectivity=0.75, seed=42)
A = net.get_adjacency_matrix()
fidelities = [node.get_purity() for node in net.nodes]
plot_network_graph(A, node_fidelities=fidelities,
                   title='Initial Network State')
plt.show()

## Phase D: Neural Noise Prediction


In [ ]:
import torch
t = np.linspace(0, 20, 1000)
noise_data = np.clip(
    0.05 + 0.03*np.sin(0.5*t) + 0.02*np.sin(2*t) + 0.01*np.random.randn(1000),
    0.001, 0.5
).astype(np.float32)

model = NoisePredictor(seq_length=20, n_features=5, hidden_dim=64, forecast_horizon=5)
trainer = NoisePredictorTrainer(model, learning_rate=5e-4)
train_result = trainer.train(noise_data, n_epochs=30, early_stopping_patience=8)

plt.figure(figsize=(10,4))
plt.semilogy(train_result['train_losses'], label='Train')
plt.semilogy(train_result['val_losses'], label='Validation')
plt.title('Training Loss')
plt.xlabel('Epoch')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()
print(f'Best val loss: {min(train_result["val_losses"]):.6f}')

## Phase E: Unified Adaptive Control


In [ ]:
net2 = HyperconnectedQuantumNetwork(n_nodes=9, seed=99)
corrector = TopologicalErrorCorrector(net2)
slime = QuantumSlimeMoldOptimizer(net2, adaptive=True)
controller = HybridAdaptiveController(
    net2, corrector, slime,
    predictor_backend='torch', seq_length=20, forecast_horizon=5
)
noise_profile = np.clip(
    0.05 + 0.03*np.sin(0.3*t[:100]) + 0.01*np.random.randn(100),
    0.001, 0.5
).astype(np.float32)
results = controller.run_full_control_loop(noise_profile, pretrain_steps=300)
print(f'Avg Fidelity: {np.mean(results["fidelity_history"]):.4f}')
print(f'Corrections:  {len(results["correction_events"])}')